In [2]:
# environment check
import torch

print("Torch version:", torch.__version__)
print("CUDA in torch:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Capability:", torch.cuda.get_device_capability(0))



Torch version: 2.9.0+cu130
CUDA in torch: 13.0
GPU Available: True
GPU: NVIDIA GeForce RTX 5080
GPU Capability: (12, 0)


In [3]:
import segmentation_models_pytorch as smp
# Use a U-Net model with VGG16 encoder
model = smp.Unet( # can use other structures like FPN, Linknet, PSPNet
    encoder_name="vgg16", # can be changed to other architectures
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
print(model)


c:\Users\18721\miniconda3\envs\416env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unet(
  (encoder): VGGEncoder(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=True)
      

In [1]:
import os
import torch
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm


base_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_after_aug")
train_img_dir = base_dir / "train" / "images"
train_mask_dir = base_dir / "train" / "labels"
val_img_dir   = base_dir / "val" / "images"
val_mask_dir  = base_dir / "val" / "labels"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30

class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform_img=None, transform_mask=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform_img = transform_img
        self.transform_mask = transform_mask
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]
        image_path = self.img_dir / image_name

        mask_name = image_name.replace(".jpg", "_mask.png") # label file naming convention
        mask_path = self.mask_dir / mask_name

        image = Image.open(image_path).convert("L")

        # Resize + ToTensor
        if self.transform_img:
            image = self.transform_img(image)

        # grey to 3-channel by repeating
        image = image.repeat(3, 1, 1)

        # Normalize to ImageNet mean/std 
        image = transforms.functional.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        # Mask
        mask = Image.open(mask_path).convert("L")
        if self.transform_mask:
            mask = self.transform_mask(mask)

        # mask = (mask > 0.5).float()  
        mask = (mask > 0).float()


        return image, mask


# Resizing to 256*256
transform_img = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

transform_mask = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=Image.NEAREST),
    transforms.ToTensor(),
])


train_ds = SegDataset(train_img_dir, train_mask_dir, transform_img, transform_mask)
val_ds = SegDataset(val_img_dir, val_mask_dir, transform_img, transform_mask)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)


# Model (UNet + VGG16 encoder frozen)

model = smp.UnetPlusPlus(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(DEVICE)

# Freeze encoder (like freezing fully-connected head removed)
for param in model.encoder.parameters():
    param.requires_grad = False

# Unfreeze last block of encoder(block5, last 6 layers)
for param in list(model.encoder.features.parameters())[-6:]:
    param.requires_grad = True




print("Trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

# loss function and optimizer
loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE) # Dice Loss for binary segmentation, shape agnostic
bce = torch.nn.BCEWithLogitsLoss() # BCE with logits, pixel-wise

def loss_function(pred, mask):
    return 0.5 * bce(pred, mask) + 0.5 * loss_fn(pred, mask)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

# train and validation 

#IoU evaluation
best_iou = 0
def iou_score(pred, mask, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * mask).sum()
    union = (pred + mask).sum() - inter
    return (inter + 1e-7) / (union + 1e-7)

# Dice evaluation
best_dice = 0
def dice_metric(pred, mask, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * mask).sum()
    return (2 * inter + 1e-7) / (pred.sum() + mask.sum() + 1e-7)


for epoch in range(EPOCHS):
    # Training
    model.train()
    model.encoder.eval() # freeze encoder during training, BN layers stay in eval mode
    train_loss = 0
    
    for img, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        
        optimizer.zero_grad() # reset gradients accumlation before each backpropagation
        pred = model(img) # forward pass
        loss = loss_function(pred, mask)
        loss.backward()
        optimizer.step() # update weights
        
        train_loss += loss.item()

    model.eval()
    val_iou = 0
    val_dice = 0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            pred = model(img)
            val_iou += iou_score(pred, mask).item()
            val_dice += dice_metric(pred, mask).item()
    
    train_loss /= len(train_loader)
    val_iou /= len(val_loader)
    val_dice /= len(val_loader) 
    
    print(f" Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | IoU: {val_iou:.4f} | Dice: {val_dice:.4f}")

    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_IoU_vgg16_unet++_30epoch_aa_en.pth")
        print("Best model saved!")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "best_dice_vgg16_unet++_30epoch_aa_en.pth")
        print("Best Dice model saved!")

print("Training Finished! Best IoU:", best_iou)
print("Best Dice:", best_dice)


c:\Users\18721\miniconda3\envs\416env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Trainable params: 31712081


Epoch 1/30: 100%|██████████| 420/420 [01:47<00:00,  3.90it/s]


 Epoch 1/30 | Loss: 0.6280 | IoU: 0.2037 | Dice: 0.2806
Best model saved!
Best Dice model saved!


Epoch 2/30: 100%|██████████| 420/420 [02:02<00:00,  3.42it/s]


...

 Epoch 28/30 | Loss: 0.0582 | IoU: 0.2617 | Dice: 0.3296


Epoch 29/30: 100%|██████████| 420/420 [02:07<00:00,  3.30it/s]


 Epoch 29/30 | Loss: 0.0651 | IoU: 0.3061 | Dice: 0.3766


Epoch 30/30: 100%|██████████| 420/420 [02:07<00:00,  3.29it/s]


 Epoch 30/30 | Loss: 0.0547 | IoU: 0.3147 | Dice: 0.3913
Training Finished! Best IoU: 0.3487825674051529
Best Dice: 0.426939664009952


In [ ]:
# visualization
model.load_state_dict(torch.load("best_dice_vgg16_unet++_30epoch_aa_en.pth", map_location=DEVICE))

model.eval()


# normalize before training, now unnormalize for visualization
def unnormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    tensor = tensor.cpu().numpy().transpose(1,2,0)
    tensor = (tensor * std + mean)
    tensor = np.clip(tensor, 0, 1)
    return tensor


num_show = 34

plt.figure(figsize=(15, 5 * num_show)) # subplot size

for idx in range(num_show):
    img, mask = val_ds[idx]  # from dataset
    img_tensor = img.unsqueeze(0).to(DEVICE)  # add batch dim

    with torch.no_grad(): # just inference
        pred = model(img_tensor)
        pred = torch.sigmoid(pred)
        pred = (pred > 0.5).float().cpu().squeeze().numpy()  # threshold to binary

    # Convert tensors for plotting
    img_np = unnormalize(img)   

    mask_np = mask.cpu().squeeze().numpy()

    # Plotting
    plt.subplot(num_show, 3, idx * 3 + 1)
    plt.imshow(img_np)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(num_show, 3, idx * 3 + 2)
    plt.imshow(mask_np, cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(num_show, 3, idx * 3 + 3)
    plt.imshow(pred, cmap="gray")
    plt.title("Predicted")
    plt.axis("off")


plt.tight_layout()
plt.show()





NameError: name 'model' is not defined